# Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set plotting style 
sns.set_style("whitegrid")

# Import functions from our src module
import sys
import os

# Add the src directory to the Python path
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '../src')))

from src.data_preparation import load_raw_data, clean_and_prepare_data

In [ ]:
# Define the path to your raw data
raw_data_path = '../data/raw/MachineLearningRating_v3.txt'

# Load and clean the data
df_raw = load_raw_data(raw_data_path)
df = clean_and_prepare_data(df_raw)

if df.empty:
    print("DataFrame is empty after loading/cleaning. Cannot proceed with EDA.")
else:
    print("\n--- Initial Glimpse of Cleaned Data ---")
    display(df.head())
    print("\n--- Cleaned Data Information ---")
    df.info()

# Data Quality Assessment 

In [ ]:
print("\n--- Missing Values Count ---")
missing_values = df.isnull().sum()
missing_values = missing_values[missing_values > 0].sort_values(ascending=False)
print(missing_values)

print("\n--- Missing Values Percentage ---")
missing_percentage = (df.isnull().sum() / len(df)) * 100
missing_percentage = missing_percentage[missing_percentage > 0].sort_values(ascending=False)
print(missing_percentage)

# Actionable Insight: Identify columns with significant missing data.
# For example, if 'PostalCode' has many NaNs, this will impact geo-based analysis.


# Data Summarization - Descriptive Statistics

In [ ]:
print("\n--- Descriptive Statistics for Numerical Columns ---")
display(df.describe())

print("\n--- Descriptive Statistics for Categorical Columns ---")
display(df.describe(include='object'))

# Actionable Insight: Check min/max values. Are there outliers (e.g., very high claims)?
# Are counts for categorical variables balanced or highly skewed?

# Univariate Analysis - Numerical Distributions (Histograms)

In [ ]:
numerical_cols = ['TotalPremium', 'TotalClaims', 'CustomValueEstimate', 'CapitalOutstanding',
                  'CalculatedPremiumPerTerm', 'Cylinders', 'cubiccapacity', 'kilowatts',
                  'NumberOfDoors', 'RegistrationYear', 'NumberOfVehiclesInFleet', 'Margin']

# Filter out columns that might not exist or are empty
numerical_cols = [col for col in numerical_cols if col in df.columns]

plt.figure(figsize=(18, 15))
for i, col in enumerate(numerical_cols):
    if i >= len(numerical_cols): # Safety break if subplot count is exceeded
        break
    plt.subplot(4, 4, i + 1)
    sns.histplot(df[col].dropna(), kde=True, bins=30)
    plt.title(f'Distribution of {col}', fontsize=10)
    plt.xlabel(col, fontsize=8)
    plt.ylabel('Frequency', fontsize=8)
    plt.tick_params(axis='both', which='major', labelsize=7)
plt.tight_layout()
plt.suptitle('Univariate Distributions of Numerical Features', y=1.02, fontsize=16)
plt.savefig('../reports/figures/numerical_distributions.png') # Save plot
plt.show()

# Actionable Insight: Note skewness, multi-modality, and ranges.
# For example, TotalClaims likely shows a highly skewed distribution with many zeros.

# Univariate Analysis - Categorical Distributions (Bar Charts)

In [ ]:
categorical_cols = ['Gender', 'Province', 'VehicleType', 'Make', 'Bodytype', 'ExcessSelected',
                    'CoverCategory', 'CoverType', 'Product', 'AlarmImmobiliser', 'TrackingDevice',
                    'NewVehicle', 'WrittenOff', 'Rebuilt', 'Converted', 'CrossBorder']

categorical_cols = [col for col in categorical_cols if col in df.columns]


plt.figure(figsize=(18, 20))
for i, col in enumerate(categorical_cols):
    if i >= len(categorical_cols): # Safety break
        break
    plt.subplot(5, 4, i + 1) # Adjust grid as needed
    sns.countplot(y=df[col], order=df[col].value_counts().index, palette='viridis')
    plt.title(f'Distribution of {col}', fontsize=10)
    plt.xlabel('Count', fontsize=8)
    plt.ylabel(col, fontsize=8)
    plt.tick_params(axis='both', which='major', labelsize=7)
plt.tight_layout()
plt.suptitle('Univariate Distributions of Categorical Features', y=1.02, fontsize=16)
plt.savefig('../reports/figures/categorical_distributions.png') # Save plot
plt.show()

# Actionable Insight: Identify dominant categories. Are some categories too rare?
# For example, if 'Make' has hundreds of unique values, you might need to group them.

# Bivariate/Multivariate Analysis - Overall Loss Ratio & Variations

In [ ]:
print("\n--- Loss Ratio Analysis ---")

# Calculate overall loss ratio
# Avoid division by zero if TotalPremium sum is 0
overall_loss_ratio = df['TotalClaims'].sum() / df['TotalPremium'].sum() if df['TotalPremium'].sum() > 0 else 0
print(f"Overall Loss Ratio (TotalClaims / TotalPremium): {overall_loss_ratio:.4f}")

# Loss Ratio by Province
province_loss_ratio = df.groupby('Province').agg(
    TotalClaims=('TotalClaims', 'sum'),
    TotalPremium=('TotalPremium', 'sum')
).reset_index()
# Avoid division by zero
province_loss_ratio['Loss_Ratio'] = province_loss_ratio.apply(
    lambda row: row['TotalClaims'] / row['TotalPremium'] if row['TotalPremium'] > 0 else 0, axis=1
)
province_loss_ratio = province_loss_ratio.sort_values(by='Loss_Ratio', ascending=False)
print("\nLoss Ratio by Province:")
display(province_loss_ratio)

plt.figure(figsize=(10, 6))
sns.barplot(x='Loss_Ratio', y='Province', data=province_loss_ratio, palette='coolwarm')
plt.title('Loss Ratio by Province')
plt.xlabel('Loss Ratio')
plt.ylabel('Province')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('../reports/figures/loss_ratio_by_province.png')
plt.show()

# Loss Ratio by VehicleType (Top N)
vehicletype_loss_ratio = df.groupby('VehicleType').agg(
    TotalClaims=('TotalClaims', 'sum'),
    TotalPremium=('TotalPremium', 'sum')
).reset_index()
vehicletype_loss_ratio['Loss_Ratio'] = vehicletype_loss_ratio.apply(
    lambda row: row['TotalClaims'] / row['TotalPremium'] if row['TotalPremium'] > 0 else 0, axis=1
)
vehicletype_loss_ratio = vehicletype_loss_ratio.sort_values(by='Loss_Ratio', ascending=False)
# Display top 10 for clarity
print("\nLoss Ratio by VehicleType (Top 10):")
display(vehicletype_loss_ratio.head(10))

# Loss Ratio by Gender
gender_loss_ratio = df.groupby('Gender').agg(
    TotalClaims=('TotalClaims', 'sum'),
    TotalPremium=('TotalPremium', 'sum')
).reset_index()
gender_loss_ratio['Loss_Ratio'] = gender_loss_ratio.apply(
    lambda row: row['TotalClaims'] / row['TotalPremium'] if row['TotalPremium'] > 0 else 0, axis=1
)
print("\nLoss Ratio by Gender:")
display(gender_loss_ratio)

plt.figure(figsize=(6, 4))
sns.barplot(x='Gender', y='Loss_Ratio', data=gender_loss_ratio, palette='pastel')
plt.title('Loss Ratio by Gender')
plt.xlabel('Gender')
plt.ylabel('Loss Ratio')
plt.tight_layout()
plt.savefig('../reports/figures/loss_ratio_by_gender.png')
plt.show()


# Actionable Insight: Provinces with high loss ratios are less profitable, indicating higher risk.
# Vehicle types with high loss ratios are also higher risk.

# Temporal Trends

In [ ]:
print("\n--- Temporal Trends Analysis ---")

# Aggregate by TransactionMonth
monthly_data = df.groupby('TransactionMonth').agg(
    TotalClaims=('TotalClaims', 'sum'),
    TotalPremium=('TotalPremium', 'sum'),
    NumPolicies=('PolicyID', 'nunique'), # Count unique policies per month
    NumClaimsOccurred=('HasClaim', 'sum') # Count policies that had a claim
).reset_index()

# Calculate Claim Frequency Rate (policies with claims / total policies)
monthly_data['Claim_Frequency_Rate'] = (monthly_data['NumClaimsOccurred'] / monthly_data['NumPolicies']).fillna(0)

# Calculate Average Claim Severity (TotalClaims / NumClaimsOccurred) for claims-only policies
# Need to join with claims_df or recalculate more carefully. For simplicity here:
# Average severity across all policies (including 0 claim policies)
monthly_data['Average_Claim_Amount_Per_Policy'] = (monthly_data['TotalClaims'] / monthly_data['NumPolicies']).fillna(0)
# If you want severity ONLY for policies with claims:
df_claims_only = df[df['TotalClaims'] > 0].copy()
monthly_severity_claims_only = df_claims_only.groupby('TransactionMonth')['TotalClaims'].mean().reset_index()
monthly_data = monthly_data.merge(monthly_severity_claims_only, on='TransactionMonth', how='left', suffixes=('', '_Severity_ClaimsOnly'))
monthly_data['TotalClaims_Severity_ClaimsOnly'] = monthly_data['TotalClaims_Severity_ClaimsOnly'].fillna(0)


plt.figure(figsize=(14, 7))
sns.lineplot(x='TransactionMonth', y='TotalPremium', data=monthly_data, label='Total Monthly Premium', marker='o')
sns.lineplot(x='TransactionMonth', y='TotalClaims', data=monthly_data, label='Total Monthly Claims', marker='o')
plt.title('Monthly Trends: Total Premium vs. Total Claims (Feb 2014 - Aug 2015)')
plt.xlabel('Month')
plt.ylabel('Amount')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.savefig('../reports/figures/monthly_premium_claims_trend.png')
plt.show()


plt.figure(figsize=(14, 7))
sns.lineplot(x='TransactionMonth', y='Claim_Frequency_Rate', data=monthly_data, label='Claim Frequency Rate', marker='o', color='green')
sns.lineplot(x='TransactionMonth', y='TotalClaims_Severity_ClaimsOnly', data=monthly_data, label='Average Claim Severity (Claims Only)', marker='o', color='purple')
plt.title('Monthly Trends: Claim Frequency Rate and Average Claim Severity')
plt.xlabel('Month')
plt.ylabel('Rate / Amount')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.savefig('../reports/figures/monthly_freq_severity_trend.png')
plt.show()

# Actionable Insight: Look for seasonality or consistent increases/decreases over the 18 months.

# Vehicle Make/Model Claim Amounts

In [ ]:
print("\n--- Vehicle Make Claim Analysis ---")

# Average claim by Make (only for policies with claims)
make_avg_claims = df_claims_only.groupby('Make')['TotalClaims'].mean().sort_values(ascending=False)

# Consider only Makes with a reasonable number of policies to avoid skewed averages from rare Makes
make_counts = df['Make'].value_counts()
min_policies_for_make_analysis = 50 # Example threshold
common_makes = make_counts[make_counts >= min_policies_for_make_analysis].index
make_avg_claims_filtered = make_avg_claims[make_avg_claims.index.isin(common_makes)]


top_n_makes = 15 # Top N for plotting
bottom_n_makes = 15 # Bottom N for plotting

print(f"\nAverage Claims by Make (Top {top_n_makes} common makes):")
display(make_avg_claims_filtered.head(top_n_makes))
print(f"\nAverage Claims by Make (Bottom {bottom_n_makes} common makes):")
display(make_avg_claims_filtered.tail(bottom_n_makes))


# Plotting top/bottom N common makes
plt.figure(figsize=(15, 8))
sns.barplot(x=make_avg_claims_filtered.head(top_n_makes).index, y=make_avg_claims_filtered.head(top_n_makes).values, palette='Reds_d')
plt.title(f'Top {top_n_makes} Common Makes by Average Claim Amount (Claims Only)')
plt.xlabel('Vehicle Make')
plt.ylabel('Average Claim Amount')
plt.xticks(rotation=60, ha='right')
plt.tight_layout()
plt.savefig('../reports/figures/top_makes_by_avg_claim.png')
plt.show()

plt.figure(figsize=(15, 8))
sns.barplot(x=make_avg_claims_filtered.tail(bottom_n_makes).index, y=make_avg_claims_filtered.tail(bottom_n_makes).values, palette='Greens_d')
plt.title(f'Bottom {bottom_n_makes} Common Makes by Average Claim Amount (Claims Only)')
plt.xlabel('Vehicle Make')
plt.ylabel('Average Claim Amount')
plt.xticks(rotation=60, ha='right')
plt.tight_layout()
plt.savefig('../reports/figures/bottom_makes_by_avg_claim.png')
plt.show()

# Actionable Insight: Identify specific car makes that are associated with higher or lower claim costs.

In [ ]:
#Outlier Detection (Box Plots for Financials)

In [ ]:
print("\n--- Outlier Detection for Financial Columns ---")

financial_cols_for_outliers = ['TotalPremium', 'TotalClaims', 'CustomValueEstimate', 'SumInsured', 'CapitalOutstanding', 'Margin']
financial_cols_for_outliers = [col for col in financial_cols_for_outliers if col in df.columns]

plt.figure(figsize=(18, 10))
for i, col in enumerate(financial_cols_for_outliers):
    if i >= len(financial_cols_for_outliers): # Safety break
        break
    plt.subplot(2, 3, i + 1)
    sns.boxplot(y=df[col])
    plt.title(f'Box Plot of {col}')
    plt.ylabel(col, fontsize=8)
    plt.tick_params(axis='y', which='major', labelsize=7)
plt.tight_layout()
plt.suptitle('Outlier Detection in Financial Features', y=1.02, fontsize=16)
plt.savefig('../reports/figures/financial_outliers_boxplots.png')
plt.show()

# Actionable Insight: Understand the range and presence of extreme values, especially in TotalClaims,
# which can heavily influence averages and models. Decide on a strategy for handling them later (capping, transformation).

In [ ]:
# Creative and Beautiful Plots (Examples)

In [ ]:
# Produce 3 creative and beautiful plots that capture key insights.
# These plots should be chosen based on the most impactful findings from your EDA.

# Example 1: Combined Loss Ratio by Province and Gender
# Using melt for easier plotting if needed, or direct barplots.

df_grouped_loss_gender_province = df.groupby(['Province', 'Gender']).agg(
    TotalClaims=('TotalClaims', 'sum'),
    TotalPremium=('TotalPremium', 'sum')
).reset_index()

df_grouped_loss_gender_province['Loss_Ratio'] = df_grouped_loss_gender_province.apply(
    lambda row: row['TotalClaims'] / row['TotalPremium'] if row['TotalPremium'] > 0 else 0, axis=1
)

plt.figure(figsize=(14, 8))
sns.barplot(x='Province', y='Loss_Ratio', hue='Gender', data=df_grouped_loss_gender_province, palette='deep')
plt.title('Loss Ratio by Province and Gender')
plt.xlabel('Province')
plt.ylabel('Loss Ratio')
plt.xticks(rotation=45, ha='right')
plt.legend(title='Gender')
plt.tight_layout()
plt.savefig('../reports/figures/loss_ratio_province_gender.png')
plt.show()


# Example 2: Distribution of Policies vs. Claims across CoverType
# This helps see where the premium comes from vs where the claims are paid.
cover_type_summary = df.groupby('CoverType').agg(
    NumPolicies=('PolicyID', 'nunique'),
    TotalClaims=('TotalClaims', 'sum'),
    TotalPremium=('TotalPremium', 'sum')
).reset_index()

cover_type_summary['Claim_Ratio_of_Total_Claims'] = cover_type_summary['TotalClaims'] / cover_type_summary['TotalClaims'].sum()
cover_type_summary['Premium_Ratio_of_Total_Premium'] = cover_type_summary['TotalPremium'] / cover_type_summary['TotalPremium'].sum()
cover_type_summary = cover_type_summary.sort_values(by='Premium_Ratio_of_Total_Premium', ascending=False)

fig, ax1 = plt.subplots(figsize=(14, 8))

sns.barplot(x='CoverType', y='Premium_Ratio_of_Total_Premium', data=cover_type_summary, color='skyblue', ax=ax1, label='Proportion of Total Premium')
ax1.set_ylabel('Proportion of Total Premium', color='skyblue')
ax1.tick_params(axis='y', labelcolor='skyblue')
ax1.set_xlabel('Cover Type')
ax1.set_title('Proportion of Total Premium vs. Claims by Cover Type')
plt.xticks(rotation=45, ha='right')


ax2 = ax1.twinx() # Create a second y-axis
sns.lineplot(x='CoverType', y='Claim_Ratio_of_Total_Claims', data=cover_type_summary, marker='o', color='red', ax=ax2, label='Proportion of Total Claims')
ax2.set_ylabel('Proportion of Total Claims', color='red')
ax2.tick_params(axis='y', labelcolor='red')

fig.legend(loc="upper right", bbox_to_anchor=(1,1), bbox_transform=ax1.transAxes)
plt.tight_layout()
plt.savefig('../reports/figures/cover_type_premium_claims_proportion.png')
plt.show()


# Example 3: Impact of 'AlarmImmobiliser' and 'TrackingDevice' on Average Claim
# Filter for claims only for severity calculation
df_security_claims = df_claims_only.copy()

# Melt the dataframe to easily plot both security features
df_melted_security = df_security_claims.melt(id_vars=['TotalClaims'], value_vars=['AlarmImmobiliser', 'TrackingDevice'], var_name='Security_Feature', value_name='Has_Feature')

plt.figure(figsize=(12, 7))
sns.boxplot(x='Security_Feature', y='TotalClaims', hue='Has_Feature', data=df_melted_security, palette='Paired')
plt.title('Impact of Security Features on Average Claim Amount (Claims Only)')
plt.xlabel('Security Feature')
plt.ylabel('Total Claim Amount')
plt.yscale('log') # Use log scale if claim amounts vary widely
plt.tight_layout()
plt.savefig('../reports/figures/security_features_claim_impact.png')
plt.show()

print("\n--- EDA complete. Check 'reports/figures/' for generated plots. ---")